# Experiment 2 — Manuscript & Supplemental Figures

Generates all figures for Experiment 2 (episodic reality monitoring).

**Outputs**
- `reports/figures/manuscript/Fig4_exp2_accuracy_rh.pdf/.png`  — accuracy + RH (main fig 4)
- `reports/figures/manuscript/Fig5_exp2_metacognition.pdf/.png` — gamma by condition (main fig 5)
- `reports/figures/manuscript/Fig6_exp2_cumulative.pdf/.png`    — cumulative accuracy (main fig 6)
- `reports/figures/supplemental/FigS4_exp2_confidence.pdf/.png` — confidence distributions
- `reports/figures/supplemental/FigS5_exp2_relatedness.pdf/.png` — relatedness ratings
- `reports/figures/supplemental/FigS6_exp2_gamma_src.pdf/.png`  — gamma by source detail

**Run with project venv:** `.venv/bin/jupyter nbconvert --to notebook --execute notebooks/05_figures/exp2_manuscript_figures.ipynb`

In [2]:
# ── 0. Imports & config ───────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from pathlib import Path

import rmllm
from rmllm import gamma as gamma_mod

PROJ = Path(rmllm.config.PROJ_ROOT)
DATA = PROJ / 'data' / 'processed'
MAN  = PROJ / 'reports' / 'figures' / 'manuscript'
SUP  = PROJ / 'reports' / 'figures' / 'supplemental'
MAN.mkdir(parents=True, exist_ok=True)
SUP.mkdir(parents=True, exist_ok=True)

DPI_MANUSCRIPT = 700

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 11,
    'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.labelsize': 10, 'axes.labelweight': 'bold',
    'xtick.labelsize': 9,  'ytick.labelsize': 9,
    'axes.linewidth': 1.0, 'axes.facecolor': 'white',
    'figure.facecolor': 'white', 'axes.grid': False,
    'xtick.bottom': True, 'ytick.left': True,
    'xtick.direction': 'out', 'ytick.direction': 'out',
    'xtick.major.size': 4, 'ytick.major.size': 4,
    'legend.fontsize': 8.5, 'legend.framealpha': 0.9,
    'legend.edgecolor': '#cccccc',
})

MODEL_ORDER  = ['Gemma3:12b','Gemma3:12b-QAT','Gemma3:27b',
                'Gemma3:27b-QAT','Llama3.3:70b','Llama4:16x17b']
MODEL_LABELS = ['G3:12b','G3:12b\nQAT','G3:27b',
                'G3:27b\nQAT','L3.3:70b','L4:16x17b']
N_MDL = len(MODEL_ORDER)
x     = np.arange(N_MDL)
bw    = 0.38

FB_PAL  = {True: '#2196F3', False: '#FF9800'}   # blue=feedback, orange=no feedback
FB_LABEL= {True: 'Feedback', False: 'No Feedback'}
SRC_PAL = {'test:perceived': '#2166ac', 'test:imagined': '#d6604d'}
SRC_LABEL = {'test:perceived': 'External', 'test:imagined': 'Internal'}
SRC_HATCH = {'test:perceived': '', 'test:imagined': '///'}

def _xticks(ax):
    ax.set_xticks(x)
    ax.set_xticklabels(MODEL_LABELS, fontsize=8.5, fontweight='bold',
                       rotation=40, ha='right', rotation_mode='anchor')
    ax.set_xlim(-0.6, N_MDL - 0.4)

def _panel_tag(ax, letter, title=''):
    ax.text(-0.13, 1.09, letter, transform=ax.transAxes,
            fontsize=13, fontweight='bold', va='top', ha='left')
    if title:
        ax.set_title(title, pad=5, fontsize=10, fontweight='bold')

print('Setup complete.')

Setup complete.


In [3]:
# ── 1. Load & preprocess data ─────────────────────────────────────────────
df = pd.read_csv(DATA / 'exp2_trial_data.csv')

# Normalise types
df['model']               = df['model'].astype(str).str.strip()
df['source_test']         = df['source_test'].astype(str).str.strip()
df['setsize']             = df['setsize'].astype(int)
df['fb_exp']              = df['fb_exp'].astype(bool)
df['accuracy']            = pd.to_numeric(df['accuracy'],            errors='coerce')
df['reading_hallucination']= pd.to_numeric(df['reading_hallucination'], errors='coerce')
df['confidence']          = pd.to_numeric(df['confidence'],          errors='coerce')
df['rating']              = pd.to_numeric(df['rating'],              errors='coerce')
df['cum_acc']             = pd.to_numeric(df['cum_acc'],             errors='coerce')
df['trial_count']         = pd.to_numeric(df['trial_count'],         errors='coerce')

# Perceived-only subset (for RH)
df_perc = df[df['source_test'] == 'test:perceived'].copy()

CONDITIONS = [(20, False), (20, True), (40, False), (40, True)]
COND_LABELS = ['20 trials\nNo FB', '20 trials\nFeedback',
               '40 trials\nNo FB', '40 trials\nFeedback']
SOURCES = ['test:imagined', 'test:perceived']

print(f'Total trials: {len(df):,}  |  models: {sorted(df.model.unique())}')
print(f'source_test: {sorted(df.source_test.unique())}')
print(f'setsize: {sorted(df.setsize.unique())}  |  fb_exp: {sorted(df.fb_exp.unique())}')

Total trials: 72,000  |  models: ['Gemma3:12b', 'Gemma3:12b-QAT', 'Gemma3:27b', 'Gemma3:27b-QAT', 'Llama3.3:70b', 'Llama4:16x17b']
source_test: ['test:imagined', 'test:perceived']
setsize: [np.int64(20), np.int64(40)]  |  fb_exp: [np.False_, np.True_]


In [4]:
# ── 2. Compute summaries ──────────────────────────────────────────────────
# Per-trace aggregation: collapse to one value per trace first, then take
# mean ± SEM across the 200 independent traces per model x condition cell.
# This matches the (1 | trace) random-effect structure used in the
# GLMMs/LMMs and the convention already used below for gamma, SDT, and
# cumulative accuracy (gamma_agg, cum) -- avoids understating uncertainty
# by treating within-trace trials as independent observations.

# Accuracy by model × source × setsize × fb_exp (per-trace, then across traces)
acc_trace = (df.groupby(['trace','model','source_test','setsize','fb_exp'], observed=True)['accuracy']
               .mean().reset_index())
acc = (acc_trace.groupby(['model','source_test','setsize','fb_exp'], observed=True)['accuracy']
                 .agg(['mean','sem']).reset_index())
acc.columns = ['model','source_test','setsize','fb_exp','acc_mean','acc_sem']

# RH by model × setsize × fb_exp (perceived only; per-trace, then across traces)
rh_trace = (df_perc.groupby(['trace','model','setsize','fb_exp'], observed=True)['reading_hallucination']
                    .mean().reset_index())
rh = (rh_trace.groupby(['model','setsize','fb_exp'], observed=True)['reading_hallucination']
               .agg(['mean','sem']).reset_index())
rh.columns = ['model','setsize','fb_exp','rh_mean','rh_sem']

# Gamma by trace × model × source × setsize × fb_exp, then mean across traces
gamma_rows = []
for (trace, mdl, src, ss, fb), grp in df.groupby(
        ['trace','model','source_test','setsize','fb_exp'], observed=True):
    if len(grp) < 3:
        continue
    g = gamma_mod.goodman_kruskal_gamma(grp['accuracy'].values, grp['confidence'].values)
    if not np.isnan(g):
        fz = np.arctanh(np.clip(g, -0.9999, 0.9999))
        gamma_rows.append({'trace':trace,'model':mdl,'source_test':src,
                           'setsize':ss,'fb_exp':fb,'gamma':g,'f_gamma':fz})
df_gamma = pd.DataFrame(gamma_rows)

gamma_agg = (df_gamma.groupby(['model','source_test','setsize','fb_exp'], observed=True)['f_gamma']
                     .agg(['mean','sem']).reset_index())
gamma_agg.columns = ['model','source_test','setsize','fb_exp','gamma_mean','gamma_sem']

# Cumulative accuracy summary by model × setsize × fb_exp × trial_count
cum = (df.groupby(['model','setsize','fb_exp','trial_count'], observed=True)['cum_acc']
         .agg(['mean','sem']).reset_index())
cum.columns = ['model','setsize','fb_exp','trial_count','cum_mean','cum_sem']

print('Summaries computed.')
print(f'  gamma traces: {len(df_gamma):,}  |  valid: {(~df_gamma.f_gamma.isna()).sum():,}')

Summaries computed.
  gamma traces: 4,410  |  valid: 4,410


In [5]:
# ── 3. Fig 4 — Accuracy (a-d) + Reading Hallucination (e-f) ───────────────
#
# Layout: 2 rows
#   Row 1 (a-d): accuracy by model × source for each of 4 conditions
#   Row 2 (e-f): RH rate by model × feedback for setsize 20 and 40

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 4, figure=fig,
                        hspace=0.60, wspace=0.38,
                        left=0.06, right=0.98, top=0.97, bottom=0.14)

panel_letters = ['a','b','c','d']

# Row 1: accuracy panels (one per condition)
for ci, (ss, fb) in enumerate(CONDITIONS):
    ax = fig.add_subplot(gs[0, ci])
    sub = acc[(acc['setsize']==ss) & (acc['fb_exp']==fb)]
    offsets = [-bw/2, bw/2]
    for i, src in enumerate(SOURCES):
        s = sub[sub['source_test']==src].set_index('model').reindex(MODEL_ORDER)
        ax.bar(x + offsets[i], s['acc_mean'].values, width=bw,
               color=SRC_PAL[src], hatch=SRC_HATCH[src],
               label=SRC_LABEL[src], alpha=0.88,
               edgecolor='white', linewidth=0.5)
        ax.errorbar(x + offsets[i], s['acc_mean'].values,
                    yerr=s['acc_sem'].values,
                    fmt='none', ecolor='#333333', elinewidth=0.9, capsize=2)
    ax.axhline(0.5, color='grey', lw=0.7, ls='--', alpha=0.5, zorder=0)
    ax.set_ylim(0, 1.12)
    _xticks(ax)
    ax.set_ylabel('Accuracy (Mean ± SEM)')
    title = f'Set-size {ss} · {"Feedback" if fb else "No Feedback"}'
    _panel_tag(ax, panel_letters[ci], title)
    if ci == 1:
        ax.legend(title='Source', fontsize=8, loc='upper left')

# Row 2: RH panels (one per setsize, feedback as color)
for ri, ss in enumerate([20, 40]):
    ax = fig.add_subplot(gs[1, ri*2 : ri*2+2])   # span 2 columns each
    sub = rh[rh['setsize']==ss]
    for i, fb in enumerate([False, True]):
        s = sub[sub['fb_exp']==fb].set_index('model').reindex(MODEL_ORDER)
        ax.bar(x + (i-0.5)*bw, s['rh_mean'].values, width=bw,
               color=FB_PAL[fb], label=FB_LABEL[fb],
               alpha=0.88, edgecolor='white', linewidth=0.5)
        ax.errorbar(x + (i-0.5)*bw, s['rh_mean'].values,
                    yerr=s['rh_sem'].values,
                    fmt='none', ecolor='#333333', elinewidth=0.9, capsize=2)
    ax.set_ylim(0, 1.05)
    _xticks(ax)
    ax.set_ylabel('Reading Hallucination Rate')
    letter = 'e' if ss==20 else 'f'
    _panel_tag(ax, letter, f'Reading Hallucinations — Set-size {ss}')
    ax.legend(title='Feedback', fontsize=8)


for fmt, dpi in [('pdf', DPI_MANUSCRIPT), ('png', DPI_MANUSCRIPT)]:
    fig.savefig(MAN / f'Fig4_exp2_accuracy_rh.{fmt}', dpi=dpi, bbox_inches='tight')
print('Fig 4 saved')
plt.close('all')

Fig 4 saved


In [6]:
# ── 4. Fig 5 — Metacognitive Sensitivity — Connected Dot Plot ─────────────
# Layout : 2×2 (rows=feedback, cols=set size)
# Within : one dot per model per source (Imagined=orange, Perceived=blue)
#          connected by a vertical grey line; ±1 SEM error bars
# Colors : Okabe-Ito colorblind-safe

SRC_STYLE_F5 = {
    'test:imagined':  dict(color='#D55E00', marker='o', label='Internal'),
    'test:perceived': dict(color='#0072B2', marker='s', label='External'),
}

fb_order  = [False, True]
ss_order  = [20, 40]
panel_ltr = [['a','b'],['c','d']]
FB_ROW_LABEL = {False: 'No Feedback', True: 'Feedback'}
SS_COL_LABEL = {20: 'Set Size 20', 40: 'Set Size 40'}

fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 2, figure=fig,
                        hspace=0.32, wspace=0.22,
                        left=0.08, right=0.96, top=0.95, bottom=0.14)

for ri, fb in enumerate(fb_order):
    for ci, ss in enumerate(ss_order):
        ax  = fig.add_subplot(gs[ri, ci])
        sub = gamma_agg[(gamma_agg['fb_exp'] == fb) & (gamma_agg['setsize'] == ss)]

        # Null reference + light band
        ax.axhline(0, color='#888888', lw=1.2, ls='--', zorder=1)
        ax.axhspan(-0.5, 0.5, color='#f7f7f7', zorder=0)

        # Connecting lines (imagined → perceived per model)
        for mi, model in enumerate(MODEL_ORDER):
            r_img  = sub[(sub['model'] == model) & (sub['source_test'] == 'test:imagined')]
            r_perc = sub[(sub['model'] == model) & (sub['source_test'] == 'test:perceived')]
            if r_img.empty or r_perc.empty:
                continue
            yi = float(r_img['gamma_mean'].iloc[0])
            yp = float(r_perc['gamma_mean'].iloc[0])
            ax.plot([x[mi], x[mi]], [yi, yp],
                    color='#aaaaaa', lw=1.8, alpha=0.7, zorder=2)

        # Dots + error bars per source
        for src, st in SRC_STYLE_F5.items():
            s = sub[sub['source_test'] == src].set_index('model').reindex(MODEL_ORDER)
            ax.errorbar(x, s['gamma_mean'].values, yerr=s['gamma_sem'].values,
                        fmt=st['marker'], color=st['color'],
                        markersize=9, markeredgecolor='white', markeredgewidth=1.0,
                        capsize=4, capthick=1.5, elinewidth=1.5,
                        label=st['label'], zorder=4)

        ax.set_xticks(x)
        ax.set_xticklabels(['G3:12b','G3:12b-QAT','G3:27b',
                             'G3:27b-QAT','L3.3:70b','L4:16x17b'],
                            fontsize=10, fontweight='bold',
                            rotation=35, ha='right', rotation_mode='anchor')
        ax.set_ylim(-5.8, 5.8)
        ax.spines[['top','right']].set_visible(False)
        ax.yaxis.grid(True, lw=0.6, color='#dddddd', zorder=0)
        ax.set_axisbelow(True)
        ax.tick_params(axis='y', labelsize=10)

        if ci == 0:
            ax.set_ylabel("Metacog. Sensitivity\n(γ Fisher's Z, Mean ± SEM)",
                          fontsize=11, fontweight='bold')
        if ri == 0:
            ax.set_title(SS_COL_LABEL[ss], fontsize=13, fontweight='bold', pad=10)
        if ci == 1:
            ax.annotate(FB_ROW_LABEL[fb], xy=(1.08, 0.5), xycoords='axes fraction',
                        fontsize=15, fontweight='bold', color='white',
                        rotation=-90, va='center', ha='center',
                        bbox=dict(boxstyle='round,pad=0.5',
                                  facecolor='#333333', edgecolor='none'))

        _panel_tag(ax, panel_ltr[ri][ci], '')

        if ri == 0 and ci == 0:
            handles = [
                plt.Line2D([0],[0], color=st['color'], marker=st['marker'],
                           markersize=9, markeredgecolor='white', lw=0,
                           label=st['label'])
                for st in SRC_STYLE_F5.values()
            ]
            ax.legend(handles=handles, fontsize=10, loc='upper left',
                      framealpha=0.9, edgecolor='#cccccc')

for fmt, dpi in [('pdf', DPI_MANUSCRIPT), ('png', DPI_MANUSCRIPT)]:
    fig.savefig(MAN / f'Fig5_exp2_metacognition.{fmt}', dpi=dpi, bbox_inches='tight')
print('Fig 5 saved')
plt.close('all')


Fig 5 saved


In [7]:
# ── 4b. Fig 5b — Metacognition Radar/Spider Chart ───────────────────────────
#
# Design: 2 subplots side by side (No Feedback | Feedback)
# Each subplot: radar/spider chart, one spoke per model (6 spokes)
# 4 overlaid polygons per subplot:
#   Imagined SS20 → dark red  #d6604d, solid
#   Imagined SS40 → light red #f4a582, dashed
#   Perceived SS20 → dark blue #4393c3, solid
#   Perceived SS40 → light blue #92c5de, dashed
# Concentric reference circles at γ = -4, -2, 0, 2, 4
# γ=0 circle drawn thicker/dashed to mark null

RADAR_COLORS = {
    ('test:imagined',  20): '#d6604d',
    ('test:imagined',  40): '#f4a582',
    ('test:perceived', 20): '#4393c3',
    ('test:perceived', 40): '#92c5de',
}
RADAR_LABELS = {
    ('test:imagined',  20): 'Internal, SS=20',
    ('test:imagined',  40): 'Internal, SS=40',
    ('test:perceived', 20): 'External, SS=20',
    ('test:perceived', 40): 'External, SS=40',
}
RADAR_LS = {
    ('test:imagined',  20): '-',
    ('test:imagined',  40): '--',
    ('test:perceived', 20): '-',
    ('test:perceived', 40): '--',
}
RADAR_ALPHA_FILL = 0.12
RADAR_ALPHA_LINE = 0.85

CONDITIONS_RADAR = [
    ('test:imagined',  20),
    ('test:imagined',  40),
    ('test:perceived', 20),
    ('test:perceived', 40),
]

# ── Radar geometry ───────────────────────────────────────────────────────
# Spokes: one per model, evenly spaced, starting from top (π/2), clockwise
N_SPOKES = N_MDL
angles = [np.pi/2 - 2*np.pi*i/N_SPOKES for i in range(N_SPOKES)]
angles_closed = angles + [angles[0]]  # close polygon

# Radial mapping: gamma space includes negative values.
# We use a signed radial system: polar r = gamma + OFFSET so γ=0 maps to r=OFFSET
# Reference circles at γ = -4, -2, 0, 2, 4
GRID_GAMMAS = [-4, -2, 0, 2, 4]
OFFSET = 4.0  # shift so γ=-4 → r=0, γ=0 → r=4, γ=4 → r=8

def gamma_to_r(g):
    return g + OFFSET

# ── Figure ───────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 7))

for pi, fb in enumerate([False, True]):
    ax = fig.add_subplot(1, 2, pi+1, projection='polar')
    ax.set_theta_direction(-1)         # clockwise
    ax.set_theta_offset(np.pi / 2)    # first spoke at top

    sub_fb = gamma_agg[gamma_agg['fb_exp'] == fb]

    # ── Draw concentric reference circles ────────────────────────────────
    theta_ring = np.linspace(0, 2*np.pi, 361)
    r_max = gamma_to_r(5.0)
    for gg in GRID_GAMMAS:
        r_circle = gamma_to_r(gg)
        if r_circle < 0:
            continue
        if gg == 0:
            ax.plot(theta_ring, np.full_like(theta_ring, r_circle),
                    color='black', lw=1.4, ls='--', alpha=0.55, zorder=2)
        else:
            ax.plot(theta_ring, np.full_like(theta_ring, r_circle),
                    color='#888888', lw=0.6, ls='-', alpha=0.40, zorder=1)
        # Label the circle at a fixed angle (top-right gap)
        label_theta = np.deg2rad(15)
        ax.text(label_theta, r_circle, f'γ={gg}',
                ha='left', va='center', fontsize=7, color='#555555',
                zorder=3)

    # ── Draw spoke lines ─────────────────────────────────────────────────
    for ang in angles:
        ax.plot([ang, ang], [0, r_max], color='#cccccc', lw=0.5, zorder=1)

    # ── Draw the 4 polygons ───────────────────────────────────────────────
    for (src, ss) in CONDITIONS_RADAR:
        # Gather values in MODEL_ORDER; fill missing with 0 (neutral = γ=0 → r=OFFSET is γ=0)
        vals = []
        for mdl in MODEL_ORDER:
            row = sub_fb[(sub_fb['model'] == mdl) &
                         (sub_fb['source_test'] == src) &
                         (sub_fb['setsize'] == ss)]
            if row.empty:
                vals.append(np.nan)
            else:
                vals.append(float(row['gamma_mean'].values[0]))

        # Convert to radial coords; clip to visible range [0, r_max]
        r_vals = [np.clip(gamma_to_r(v), 0, r_max) if not np.isnan(v) else None
                  for v in vals]

        # Build closed arrays (skip if any None — plot segment-by-segment)
        # For simplicity: replace None with OFFSET (γ=0) so polygon still closes
        r_clean = [gamma_to_r(v) if not np.isnan(v) else gamma_to_r(0.0) for v in vals]
        r_clean_closed = r_clean + [r_clean[0]]
        theta_closed = angles_closed

        color = RADAR_COLORS[(src, ss)]
        ls    = RADAR_LS[(src, ss)]
        label = RADAR_LABELS[(src, ss)]

        ax.plot(theta_closed, r_clean_closed,
                color=color, lw=1.8, ls=ls, alpha=RADAR_ALPHA_LINE,
                zorder=4, label=label)
        ax.fill(theta_closed, r_clean_closed,
                color=color, alpha=RADAR_ALPHA_FILL, zorder=3)

        # ── Annotate vertex values ────────────────────────────────────────
        for i, (ang, v) in enumerate(zip(angles, vals)):
            if np.isnan(v):
                continue
            r_v = gamma_to_r(v)
            r_txt = r_v + 0.45
            ax.text(ang, r_txt, f'{v:.1f}',
                    ha='center', va='center',
                    fontsize=6.5, color=color, zorder=5,
                    fontweight='bold')

    # ── Spoke labels (model names) at outer edge ──────────────────────────
    r_label = r_max + 0.85
    for i, (ang, lbl) in enumerate(zip(angles, MODEL_LABELS)):
        ax.text(ang, r_label, lbl,
                ha='center', va='center',
                fontsize=8, fontweight='bold',
                color='#222222', zorder=6)

    # ── Polar axes cosmetics ─────────────────────────────────────────────
    ax.set_ylim(0, r_max + 1.8)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines['polar'].set_visible(False)
    ax.set_title(f"{'No Feedback' if not fb else 'Feedback'}",
                 fontsize=11, fontweight='bold', pad=20)

# ── Legend below both subplots ───────────────────────────────────────────
legend_handles = []
for (src, ss) in CONDITIONS_RADAR:
    patch = mpatches.Patch(color=RADAR_COLORS[(src, ss)],
                           label=RADAR_LABELS[(src, ss)],
                           alpha=0.85)
    legend_handles.append(patch)
# Add γ=0 reference line handle
legend_handles.append(
    Line2D([0], [0], color='black', lw=1.4, ls='--', alpha=0.55,
           label='γ = 0 (null)')
)

fig.legend(handles=legend_handles,
           loc='lower center', ncol=5,
           bbox_to_anchor=(0.5, -0.06),
           fontsize=9, framealpha=0.9,
           edgecolor='#cccccc',
           title='Source × Set-size',
           title_fontsize=9)

plt.subplots_adjust(left=0.05, right=0.95, top=0.93, bottom=0.10, wspace=0.35)

for fmt, dpi in [('pdf', DPI_MANUSCRIPT), ('png', DPI_MANUSCRIPT)]:
    fig.savefig(MAN / f'Fig5b_exp2_metacognition_circular.{fmt}',
                dpi=dpi, bbox_inches='tight')
print('Fig 5b (radar) saved')
plt.close('all')


Fig 5b (radar) saved


In [8]:
# ── 5. Supplementary Figure S11 — Cumulative Accuracy Trajectories ────────
#
# Layout: 2 rows (setsize 20/40) × 6 cols (one per model)
# Within each panel: blue=no feedback, green=feedback; x=trial number

FB_LINE_PAL = {False: '#FF9800', True: '#2196F3'}   # orange=no-fb, blue=fb

fig, axes = plt.subplots(2, N_MDL, figsize=(18, 7),
                          sharex=False, sharey=True)

for ri, ss in enumerate([20, 40]):
    # Number of test trials = half the set size
    n_test = ss // 2
    sub_ss = cum[cum['setsize'] == ss]
    for ci, mdl in enumerate(MODEL_ORDER):
        ax = axes[ri, ci]
        sub = sub_ss[sub_ss['model'] == mdl]
        for fb in [False, True]:
            s = sub[sub['fb_exp'] == fb].sort_values('trial_count')
            if s.empty:
                continue
            # Keep only test-phase trial indices (1..n_test)
            s = s[s['trial_count'].between(1, n_test)]
            if s.empty:
                continue
            ax.plot(s['trial_count'], s['cum_mean'],
                    color=FB_LINE_PAL[fb], lw=2.0, alpha=0.9,
                    label=FB_LABEL[fb])
            ax.fill_between(s['trial_count'],
                            s['cum_mean'] - s['cum_sem'],
                            s['cum_mean'] + s['cum_sem'],
                            color=FB_LINE_PAL[fb], alpha=0.18)
        ax.axhline(0.5, color='grey', lw=0.7, ls='--', alpha=0.5, zorder=0)
        ax.set_ylim(0, 1.05)
        ax.set_xlim(0.5, n_test + 0.5)
        ax.set_xticks(range(1, n_test+1, max(1, n_test//5)))
        ax.tick_params(axis='x', labelsize=8)
        ax.tick_params(axis='y', labelsize=8)
        if ri == 0:
            ax.set_title(MODEL_LABELS[ci].replace('\n',' '),
                         fontsize=9, fontweight='bold')
        if ci == 0:
            ax.set_ylabel(f'Set-size {ss}\nCumulative Acc.', fontsize=9)
        if ri == 1 and ci == N_MDL // 2:
            ax.set_xlabel('Test Trial Number', fontsize=9)
        if ri == 0 and ci == 0:
            ax.legend(fontsize=7.5, loc='lower right')

plt.tight_layout()
for fmt, dpi in [('pdf', DPI_MANUSCRIPT), ('png', DPI_MANUSCRIPT)]:
    # Filename matches supp.tex's \includegraphics{FigS11_exp2_cumulative} --
    # this content is Supplementary Figure S11 (renders as S6 after the
    # 2026-07-16 renumbering), not main-text Figure 6; kept in MAN/ dir to
    # match its existing on-disk location.
    fig.savefig(MAN / f'FigS11_exp2_cumulative.{fmt}', dpi=dpi, bbox_inches='tight')
print('FigS11_exp2_cumulative saved')
plt.close('all')

Fig 6 saved


### 5b. Trajectory summary statistics (verifies FigS11 / manuscript §"Cumulative Accuracy: Heterogeneous Trial-Level Trajectories")

For each model × set size × feedback condition, computes the start (mean of first 3 test trials),
end (mean of last 3 test trials), net change, and the Pearson correlation between trial number and
cumulative accuracy (a simple monotonic-trend indicator; strong |r| = steady rise/decline, weak |r|
with small net change = flat or noisy, weak |r| with large net change or a mid-sequence trough/peak
= non-monotonic e.g. dip-then-recover). This table is the quantitative basis for every specific
model claim in the FigS11 caption and the corresponding main-text Results subsection — written to
replace an earlier draft of both passages that mischaracterized several trajectories on visual
inspection alone (e.g. citing Gemma3:12b-QAT as "stable" when it is the steepest decliner, and
Llama3.3:70b as "the largest model" when Llama4:16x17b is).

In [ ]:
traj_rows = []
for mdl in MODEL_ORDER:
    for ss in [20, 40]:
        n_test = ss // 2
        for fb in [False, True]:
            s = cum[(cum['model'] == mdl) & (cum['setsize'] == ss) & (cum['fb_exp'] == fb)]
            s = s[s['trial_count'].between(1, n_test)].sort_values('trial_count')
            if s.empty:
                continue
            start = s['cum_mean'].iloc[:3].mean()
            end   = s['cum_mean'].iloc[-3:].mean()
            corr  = s['trial_count'].corr(s['cum_mean'])
            traj_rows.append(dict(model=mdl, setsize=ss, fb_exp=fb,
                                   start=start, end=end, net_change=end - start, r=corr))

traj_summary = pd.DataFrame(traj_rows)
traj_summary['fb_label'] = traj_summary['fb_exp'].map({False: 'No-FB', True: 'FB'})
pd.set_option('display.float_format', '{:.3f}'.format)
print("=== Cumulative accuracy trajectory summary (start/end/net change/trend) ===")
print(traj_summary[['model', 'fb_label', 'setsize', 'start', 'end', 'net_change', 'r']]
      .to_string(index=False))

In [9]:
# ── 6. Supp Fig S4 — Confidence distributions ────────────────────────────
conf_levels = sorted(df['confidence'].dropna().astype(int).unique())
COLS_MDL = plt.cm.tab10.colors

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, sharey=True)

for ri, ss in enumerate([20, 40]):
    for ci, fb in enumerate([False, True]):
        ax = axes[ri, ci]
        sub = df[(df['setsize']==ss) & (df['fb_exp']==fb)]
        for mi, mdl in enumerate(MODEL_ORDER):
            grp = sub[sub['model']==mdl]
            if grp.empty:
                continue
            p = (grp['confidence'].dropna().astype(int)
                 .value_counts(normalize=True)
                 .reindex(conf_levels, fill_value=0.0)
                 .sort_index())
            ax.plot(p.index, p.values, 'o-',
                    color=COLS_MDL[mi], lw=1.6, ms=5, alpha=0.85,
                    label=MODEL_LABELS[mi].replace('\n',' '))
        ax.set_title(f'Set-size {ss} · {FB_LABEL[fb]}',
                     fontsize=10, fontweight='bold')
        ax.set_xticks(conf_levels)
        ax.set_xlabel('Confidence Level')
        ax.set_ylabel('Proportion')
        if ri==0 and ci==0:
            ax.legend(fontsize=8, ncol=2, loc='upper left')

plt.tight_layout()
for fmt, dpi in [('pdf', DPI_MANUSCRIPT), ('png', DPI_MANUSCRIPT)]:
    fig.savefig(SUP / f'FigS4_exp2_confidence.{fmt}', dpi=dpi, bbox_inches='tight')
print('FigS4 saved')
plt.close('all')

FigS4 saved


In [10]:
# ── 7. Supp Fig S5 — Relatedness ratings ─────────────────────────────────
# Per-trace aggregation (see Section 2): one value per trace, then mean±SEM
# across the 200 traces per model x condition cell.
rr_trace = (df.groupby(['trace','model','source_test','setsize','fb_exp'], observed=True)['rating']
              .mean().reset_index())
rr = (rr_trace.groupby(['model','source_test','setsize','fb_exp'], observed=True)['rating']
               .agg(['mean','sem']).reset_index())
rr.columns = ['model','source_test','setsize','fb_exp','rr_mean','rr_sem']

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=True)

for ri, ss in enumerate([20, 40]):
    for ci, fb in enumerate([False, True]):
        ax = axes[ri, ci]
        sub = rr[(rr['setsize']==ss) & (rr['fb_exp']==fb)]
        for src in SOURCES:
            s = sub[sub['source_test']==src].set_index('model').reindex(MODEL_ORDER)
            ax.errorbar(x, s['rr_mean'].values, yerr=s['rr_sem'].values,
                        color=SRC_PAL[src], lw=2.0, marker='o', ms=5,
                        capsize=3, alpha=0.9, label=SRC_LABEL[src])
        _xticks(ax)
        ax.set_ylabel('Relatedness Rating (Mean ± SEM)')
        ax.set_title(f'Set-size {ss} · {FB_LABEL[fb]}',
                     fontsize=10, fontweight='bold')
        if ri==0 and ci==0:
            ax.legend(title='Source')

plt.tight_layout()
for fmt, dpi in [('pdf', DPI_MANUSCRIPT), ('png', DPI_MANUSCRIPT)]:
    fig.savefig(SUP / f'FigS5_exp2_relatedness.{fmt}', dpi=dpi, bbox_inches='tight')
print('FigS5 saved')
plt.close('all')

FigS5 saved


In [11]:
# ── 8. Supp Fig S6 — Gamma by source × setsize × feedback detail ──────────
# 2×4 grid: rows=setsize, cols=4 conditions; split perceived/imagined bars per model

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=True)

for ri, ss in enumerate([20, 40]):
    for ci, fb in enumerate([False, True]):
        ax = axes[ri, ci]
        sub = gamma_agg[(gamma_agg['setsize']==ss) & (gamma_agg['fb_exp']==fb)]
        for i, src in enumerate(src_order := ['test:perceived','test:imagined']):
            s = sub[sub['source_test']==src].set_index('model').reindex(MODEL_ORDER)
            ax.bar(x + (i-0.5)*bw, s['gamma_mean'].values, width=bw,
                   color=SRC_PAL[src], label=SRC_LABEL[src],
                   alpha=0.88, edgecolor='white', lw=0.5)
            ax.errorbar(x + (i-0.5)*bw, s['gamma_mean'].values,
                        yerr=s['gamma_sem'].values,
                        fmt='none', ecolor='#333333', elinewidth=0.9, capsize=2)
        ax.axhline(0, color='black', lw=0.9, ls='--', alpha=0.55)
        _xticks(ax)
        ax.set_ylabel("γ (Fisher's Z)")
        ax.set_title(f'Set-size {ss} · {FB_LABEL[fb]}',
                     fontsize=10, fontweight='bold')
        if ri==0 and ci==0:
            ax.legend(title='Source', fontsize=8)

plt.tight_layout()
for fmt, dpi in [('pdf', DPI_MANUSCRIPT), ('png', DPI_MANUSCRIPT)]:
    fig.savefig(SUP / f'FigS6_exp2_gamma_by_source.{fmt}', dpi=dpi, bbox_inches='tight')
print('FigS6 saved')
plt.close('all')

FigS6 saved


In [12]:
# ── 9. Summary ────────────────────────────────────────────────────────────
print('\n=== Exp 2 Figure Notebook Complete ===')
print('Manuscript:', sorted(str(p) for p in MAN.glob('Fig[456]*')))
print('Supplemental:', sorted(str(p) for p in SUP.glob('FigS[456]*exp2*')))


=== Exp 2 Figure Notebook Complete ===
Manuscript: ['/sessions/magical-wizardly-bohr/mnt/rmllm/reports/figures/manuscript/Fig4_exp2_accuracy_rh.pdf', '/sessions/magical-wizardly-bohr/mnt/rmllm/reports/figures/manuscript/Fig4_exp2_accuracy_rh.png', '/sessions/magical-wizardly-bohr/mnt/rmllm/reports/figures/manuscript/Fig5_exp2_metacognition.pdf', '/sessions/magical-wizardly-bohr/mnt/rmllm/reports/figures/manuscript/Fig5_exp2_metacognition.png', '/sessions/magical-wizardly-bohr/mnt/rmllm/reports/figures/manuscript/Fig5b_exp2_metacognition_circular.pdf', '/sessions/magical-wizardly-bohr/mnt/rmllm/reports/figures/manuscript/Fig5b_exp2_metacognition_circular.png', '/sessions/magical-wizardly-bohr/mnt/rmllm/reports/figures/manuscript/Fig6_exp2_cumulative.pdf', '/sessions/magical-wizardly-bohr/mnt/rmllm/reports/figures/manuscript/Fig6_exp2_cumulative.png']
Supplemental: ['/sessions/magical-wizardly-bohr/mnt/rmllm/reports/figures/supplemental/FigS4_exp2_confidence.pdf', '/sessions/magical-wiz